In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('relationships_modeified1.csv')

In [3]:
df['EmpJobRole'].value_counts()

EmpJobRole
Sales Executive              270
Developer                    236
Manager R&D                   94
Research Scientist            77
Sales Representative          69
Laboratory Technician         64
Senior Developer              52
Manager                       51
Finance Manager               49
Human Resources               45
Technical Lead                38
Manufacturing Director        33
Healthcare Representative     33
Data Scientist                20
Research Director             19
Business Analyst              16
Senior Manager R&D            15
Delivery Manager              12
Technical Architect            7
Name: count, dtype: int64

In [3]:
df['EmpDepartment'].unique()

array(['Sales', 'Human Resources', 'Development', 'Data Science',
       'Research & Development', 'Finance'], dtype=object)

In [4]:
df.columns

Index(['EmpNumber', 'Age', 'Gender', 'EducationBackground', 'MaritalStatus',
       'EmpDepartment', 'EmpJobRole', 'BusinessTravelFrequency',
       'DistanceFromHome', 'EmpEducationLevel', 'EmpEnvironmentSatisfaction',
       'EmpHourlyRate', 'EmpJobInvolvement', 'EmpJobLevel',
       'EmpJobSatisfaction', 'NumCompaniesWorked', 'OverTime',
       'EmpLastSalaryHikePercent', 'EmpRelationshipSatisfaction',
       'TotalWorkExperienceInYears', 'TrainingTimesLastYear',
       'EmpWorkLifeBalance', 'ExperienceYearsAtThisCompany',
       'ExperienceYearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager', 'Attrition', 'PerformanceScore'],
      dtype='object')

In [3]:
import joblib

In [4]:
model = joblib.load("xgb_model.pkl")
encoder = joblib.load("pipeline.joblib")  # your encoder

In [5]:
df1 = df.drop(['EmpNumber','Attrition'],axis=1)

In [6]:
X_encoded = encoder.transform(df1)

In [7]:
pred = model.predict(X_encoded)

In [4]:
g = df.groupby('EmpDepartment')

In [8]:
g['PerformanceScore'].describe()

,count,mean,std,min,25%,50%,75%,max
EmpDepartment,,,,,,,,
Data Science,20.0,75.553141,2.991248,71.234386,73.428869,75.094561,78.316646,80.840211
Development,361.0,75.766318,3.984498,66.333504,72.966472,75.044590,78.257097,91.050222
Finance,49.0,75.526510,4.806728,65.687326,72.158804,74.399458,78.923994,86.691919
Human Resources,54.0,74.755607,3.636436,67.192650,72.411888,74.523656,76.870995,82.060877
Research & Development,343.0,75.618750,4.126619,65.327904,72.859667,75.137132,78.196165,88.421953
Sales,373.0,75.521929,4.101758,65.685433,72.639593,75.133243,77.647239,91.777958


In [9]:
df['PerformanceScore'].describe()

count    1200.000000
mean       75.589347
std         4.066685
min        65.327904
25%        72.808930
50%        75.057581
75%        77.984724
max        91.777958
Name: PerformanceScore, dtype: float64

In [3]:
df1 = df.copy()

In [4]:
df1.drop(['Attrition','EmpNumber'],axis=1,inplace=True)

In [5]:
x = df1.drop('PerformanceScore',axis=1)
y = df1['PerformanceScore']

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [9]:
cat_cols = x.select_dtypes('object').columns

In [10]:
num_cols = x.select_dtypes('int64').columns

In [11]:
pipeline = ColumnTransformer([
    ('onehot',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),cat_cols)
],remainder='passthrough')

In [12]:
pipeline.fit(x,y)

C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


ColumnTransformer(remainder='passthrough',
                  transformers=[('onehot',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 Index(['Gender', 'EducationBackground', 'MaritalStatus', 'EmpDepartment',
       'EmpJobRole', 'BusinessTravelFrequency', 'OverTime'],
      dtype='object'))])

In [13]:
import joblib

In [14]:
joblib.dump(pipeline,'pipeline.joblib')

['pipeline.joblib']

In [76]:
X_train_prep = pipeline.transform(X_train)
X_test_prep = pipeline.transform(X_test)

In [80]:
from xgboost import XGBRegressor

In [82]:
model_xgb = XGBRegressor(
    colsample_bytree=0.8,
    learning_rate=0.1,
    max_depth=3,
    n_estimators=400,
    subsample=0.8
)

model_xgb.fit(X_train_prep,y_train)
model_xgb.score(X_test_prep,y_test)

0.9399201076728704

In [72]:
import pickle

In [83]:
with open ('xgb_model.pkl','wb') as f:
    pickle.dump(model_xgb,f)

In [84]:
with open ('xgb_model.pkl','rb') as f:
    model = pickle.load(f)

In [85]:
model.score(X_test_prep,y_test)

0.9399201076728704

In [91]:
feature_names = pipeline.get_feature_names_out()
clean_feature_names = [name.split('__')[1] for name in feature_names]

In [92]:
clean_feature_names

['Gender_Male',
 'EducationBackground_Marketing',
 'EducationBackground_Medical',
 'EducationBackground_Other',
 'EducationBackground_Science',
 'EducationBackground_Technical Degree',
 'MaritalStatus_Married',
 'MaritalStatus_Single',
 'EmpDepartment_Development',
 'EmpDepartment_Finance',
 'EmpDepartment_Human Resources',
 'EmpDepartment_Research & Development',
 'EmpDepartment_Sales',
 'EmpJobRole_Data Scientist',
 'EmpJobRole_Delivery Manager',
 'EmpJobRole_Developer',
 'EmpJobRole_Finance Manager',
 'EmpJobRole_Healthcare Representative',
 'EmpJobRole_Human Resources',
 'EmpJobRole_Laboratory Technician',
 'EmpJobRole_Manager',
 'EmpJobRole_Manager R&D',
 'EmpJobRole_Manufacturing Director',
 'EmpJobRole_Research Director',
 'EmpJobRole_Research Scientist',
 'EmpJobRole_Sales Executive',
 'EmpJobRole_Sales Representative',
 'EmpJobRole_Senior Developer',
 'EmpJobRole_Senior Manager R&D',
 'EmpJobRole_Technical Architect',
 'EmpJobRole_Technical Lead',
 'BusinessTravelFrequency_Tra

In [94]:
import json

In [95]:
with open ('model_columns.json','w') as f:
    json.dump(clean_feature_names,f)

In [98]:
with open ('model_columns.json','r') as f:
    data_columns = json.load(f)

In [97]:
import numpy as np

In [119]:
def estimate_performance(Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,OverTime,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager):
    X = np.zeros(len(data_columns))

    fields = {
        'Gender': Gender,
        'EducationBackground':EducationBackground,
        'MaritalStatus': MaritalStatus,
        'EmpDepartment': EmpDepartment,
        'EmpJobRole': EmpJobRole,
        'BusinessTravelFrequency': BusinessTravelFrequency,
        'OverTime': OverTime
    }

    for feature,value in fields.items():
        column_name = f"{feature}_{value}"
        if column_name in data_columns:
            index = data_columns.index(column_name)
            X[index] = 1

    X[data_columns.index('Age')] = Age
    X[data_columns.index('DistanceFromHome')] = DistanceFromHome
    X[data_columns.index('EmpEducationLevel')] = EmpEducationLevel
    X[data_columns.index('EmpEnvironmentSatisfaction')] = EmpEnvironmentSatisfaction
    X[data_columns.index('EmpHourlyRate')] = EmpHourlyRate
    X[data_columns.index('EmpJobInvolvement')] = EmpJobInvolvement
    X[data_columns.index('EmpJobLevel')] = EmpJobLevel
    X[data_columns.index('EmpJobSatisfaction')] = EmpJobSatisfaction
    X[data_columns.index('NumCompaniesWorked')] = NumCompaniesWorked
    X[data_columns.index('EmpLastSalaryHikePercent')] = EmpLastSalaryHikePercent
    X[data_columns.index('EmpRelationshipSatisfaction')] = EmpRelationshipSatisfaction
    X[data_columns.index('TotalWorkExperienceInYears')] = TotalWorkExperienceInYears
    X[data_columns.index('TrainingTimesLastYear')] = TrainingTimesLastYear
    X[data_columns.index('EmpWorkLifeBalance')] = EmpWorkLifeBalance
    X[data_columns.index('ExperienceYearsAtThisCompany')] = ExperienceYearsAtThisCompany
    X[data_columns.index('ExperienceYearsInCurrentRole')] = ExperienceYearsInCurrentRole
    X[data_columns.index('YearsSinceLastPromotion')] = YearsSinceLastPromotion
    X[data_columns.index('YearsWithCurrManager')] = YearsWithCurrManager

    prediction = model.predict([X])[0]
    return prediction

In [107]:
X_test.iloc[0]

Age                                            22
Gender                                       Male
EducationBackground                       Medical
MaritalStatus                             Married
EmpDepartment                         Development
EmpJobRole                              Developer
BusinessTravelFrequency         Travel_Frequently
DistanceFromHome                                6
EmpEducationLevel                               1
EmpEnvironmentSatisfaction                      1
EmpHourlyRate                                  69
EmpJobInvolvement                               3
EmpJobLevel                                     1
EmpJobSatisfaction                              3
NumCompaniesWorked                              0
OverTime                                       No
EmpLastSalaryHikePercent                       20
EmpRelationshipSatisfaction                     4
TotalWorkExperienceInYears                      3
TrainingTimesLastYear                           3


In [114]:
y_test.iloc[0]

np.float64(72.98703088934286)

In [120]:
estimate_performance(22,'Male','Medical','Married','Development','Developer','Travel_Frequently',6,1,1,69,3,1,3,0,'No',20,4,3,3,3,2,2,2,2)

np.float32(73.31936)